In [ ]:
"""
Gaussian Process Regression (GPR)
for Predicting Compressive Strength
Author: Haseeb Ahmad

Description:
This script trains a Gaussian Process Regression model
with kernel optimization and uncertainty quantification.
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C, WhiteKernel
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.inspection import permutation_importance

import shap


# ==============================
# Configuration
# ==============================

DATA_PATH = "../data/gypsum_data.csv"
FIGURE_PATH = "../figures"
TEST_SIZE = 0.2
RANDOM_STATE = 42
N_RESTARTS = 10

os.makedirs(FIGURE_PATH, exist_ok=True)
np.random.seed(RANDOM_STATE)


# ==============================
# Load Data
# ==============================

def load_data(path):
    df = pd.read_csv(path, header=2)

    X = df.iloc[:, [4, 5, 6, 7, 8, 9, 10]].astype(float)
    y = df.iloc[:, 13].astype(float)

    data = pd.concat([X, y], axis=1).dropna()

    X_clean = data.iloc[:, :-1].values
    y_clean = data.iloc[:, -1].values

    return X_clean, y_clean


# ==============================
# Kernel Optimization
# ==============================

def optimize_kernel(X_train, y_train):

    kernel = (
        C(1.0, (1e-3, 1e3))
        * RBF(length_scale=1.0, length_scale_bounds=(1e-2, 1e2))
        + WhiteKernel(noise_level=0.1)
    )

    lml_values = []
    best_model = None
    best_lml = -np.inf

    print("\nOptimizing GPR Kernel Across Restarts...\n")

    for i in range(N_RESTARTS):
        model = GaussianProcessRegressor(
            kernel=kernel,
            optimizer="fmin_l_bfgs_b",
            random_state=RANDOM_STATE + i
        )

        model.fit(X_train, y_train)

        lml = model.log_marginal_likelihood(model.kernel_.theta)
        lml_values.append(lml)

        print(f"Restart {i+1}: LML = {lml:.4f}")
        print(f"Kernel: {model.kernel_}\n")

        if lml > best_lml:
            best_lml = lml
            best_model = model

    return best_model, lml_values


# ==============================
# Evaluation
# ==============================

def evaluate_model(y_true, y_pred):

    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)

    print("\nGPR MODEL PERFORMANCE")
    print("=" * 40)
    print(f"MSE  : {mse:.4f}")
    print(f"MAE  : {mae:.4f}")
    print(f"RMSE : {rmse:.4f}")
    print(f"R²   : {r2:.4f}")
    print("=" * 40)

    return mse, mae, rmse, r2


# ==============================
# Plotting Functions
# ==============================

def plot_lml(lml_values):

    plt.figure(figsize=(5, 4))
    plt.plot(range(1, len(lml_values) + 1), lml_values, marker="o")
    plt.xlabel("Optimizer Restart Number")
    plt.ylabel("Log-Marginal Likelihood")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURE_PATH, "gpr_lml_convergence.png"), dpi=300)
    plt.show()


def plot_actual_vs_predicted(y_true, y_pred):

    x_vals = np.linspace(y_true.min(), y_true.max(), 200)

    plt.figure(figsize=(5, 4))
    plt.scatter(y_true, y_pred, alpha=0.7, edgecolors='k')
    plt.plot(x_vals, x_vals, 'r--', label="Perfect Fit")

    plt.xlabel("Actual Compressive Strength (MPa)")
    plt.ylabel("Predicted Compressive Strength (MPa)")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURE_PATH, "gpr_actual_vs_predicted.png"), dpi=300)
    plt.show()


def plot_confidence_intervals(y_true, y_pred, sigma):

    indices = np.arange(len(y_true))

    upper = y_pred + 1.96 * sigma
    lower = y_pred - 1.96 * sigma

    plt.figure(figsize=(6, 4))
    plt.fill_between(indices, lower, upper, alpha=0.3, label="95% Confidence Interval")
    plt.plot(indices, y_true, 'o-', label="Actual")
    plt.plot(indices, y_pred, 'x--', label="Predicted")

    plt.xlabel("Sample Index")
    plt.ylabel("Compressive Strength (MPa)")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURE_PATH, "gpr_confidence_interval.png"), dpi=300)
    plt.show()


# ==============================
# Feature Importance
# ==============================

def plot_feature_importance(model, X_test, y_test, feature_names):

    result = permutation_importance(
        model,
        X_test,
        y_test,
        n_repeats=20,
        scoring="neg_mean_squared_error",
        random_state=RANDOM_STATE
    )

    importances = np.abs(result.importances_mean)
    indices = np.argsort(importances)

    plt.figure(figsize=(6, 4))
    plt.barh(range(len(importances)), importances[indices])
    plt.yticks(range(len(importances)), np.array(feature_names)[indices])
    plt.xlabel("Permutation Importance")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURE_PATH, "gpr_feature_importance.png"), dpi=300)
    plt.show()


# ==============================
# SHAP Analysis
# ==============================

def shap_analysis(model, X_train, X_test, feature_names):

    print("\nRunning SHAP KernelExplainer...\n")

    background_size = min(100, X_train.shape[0])
    background = X_train[
        np.random.choice(X_train.shape[0], background_size, replace=False)
    ]

    explainer = shap.KernelExplainer(model.predict, background)
    shap_values = explainer.shap_values(X_test)

    plt.figure(figsize=(6, 4))
    shap.summary_plot(
        shap_values,
        X_test,
        feature_names=feature_names,
        show=False
    )
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURE_PATH, "gpr_shap_summary.png"), dpi=300)
    plt.show()

    return shap_values


# ==============================
# Main
# ==============================

def main():

    X, y = load_data(DATA_PATH)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Optimize Kernel
    best_model, lml_values = optimize_kernel(X_train_scaled, y_train)

    # Plot convergence
    plot_lml(lml_values)

    # Predictions with uncertainty
    y_pred, sigma = best_model.predict(X_test_scaled, return_std=True)

    # Evaluate
    evaluate_model(y_test, y_pred)

    # Plots
    plot_actual_vs_predicted(y_test, y_pred)
    plot_confidence_intervals(y_test, y_pred, sigma)

    feature_names = [
        "Gypsum Strength",
        "Gypsum Quantity",
        "Water Quantity",
        "Water/Gypsum Ratio",
        "Wheat Straw",
        "CaCl2",
        "Ca(OH)2"
    ]

    plot_feature_importance(best_model, X_test_scaled, y_test, feature_names)

    # SHAP (optional but powerful)
    shap_analysis(best_model, X_train_scaled, X_test_scaled, feature_names)


if __name__ == "__main__":
    main()